In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install flowio

In [ ]:
import shutil
import pandas as pd
import random
import copy
import numpy as np # Import numpy for checking finite values
import matplotlib.pyplot as plt # Import matplotlib for potential debugging
import os
import math

PYTHONHASHSEED = '42'
TF_DETERMINISTIC_OPS = '1'
TF_CUDNN_DETERMINISTIC = '1'
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'  # For CuDNN ops

import glob
import json
import tensorflow as tf
import sys
from itertools import zip_longest
import pickle
import time
import csv
import traceback
import gc

from sklearn import metrics
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

In [ ]:

fixed_path = '/content/drive/MyDrive/0.Master_Thesis/'

if fixed_path not in sys.path:
    sys.path.append(fixed_path)

cellcnn_path = f'{fixed_path}CellCNN/'
if cellcnn_path not in sys.path:
    sys.path.append(cellcnn_path)

model_path = f'{cellcnn_path}Old_CellCNN/'
if model_path not in sys.path:
    sys.path.append(model_path)

save_path = f'{cellcnn_path}results/'
if save_path not in sys.path:
    sys.path.append(save_path)

modules_dir = f'{cellcnn_path}modules/'
if modules_dir not in sys.path:
    sys.path.append(modules_dir)

In [ ]:
decache_files = ['timepoints_elaboration', 'model_grid', 'run_models', 'new_datasets_generation',
                'training', 'utils', 'cv_folds', 'classification']

# Remove from cache
from utils import remove_from_cache
remove_from_cache(decache_files)

from model_grid import CellCnn

from timepoints_elaboration import load_data, donation_extraction
from run_models import  trials_train_CellCNN_old, trials_test_CellCNN_old

from new_datasets_generation import splitting_and_dataset_elaboration


from training import run_training, val_res_pred, train_val_finalizing, test_res_pred, find_theta_best
from utils import flatten, remove_labels, retrieve_labels, show_blast_distribution_perc
from utils import prepare_results_to_save, subset_sampling, generate_seeds
from utils import nsub_ncells_comb, save_models
from cv_folds import generate_LOPOCV_dicts, generate_LOPOCV_folds, extract_fold_features
from classification import find_robust_threshold

In [ ]:

tuning_exp = 'Trial_5_fix_code_NO_AS_bayesian_tuning'


config_save_dir = f'{cellcnn_path}/experiments/experiment_{tuning_exp}/'
os.makedirs(config_save_dir, exist_ok=True)

with open(os.path.join(config_save_dir, 'config.pkl'), 'rb') as f:
            config = pickle.load(f)

starting_seed = config['starting_seed']
n_sub = config['n_sub ']
n_cells = config['n_cells']
blast_perc = config['blast_perc']
nfilter = config['nfilter']
maxpool_p = config['maxpool_p']
learning_r = config['learning_r']
labels = config['labels']  # if False internal data augmentation do not takes in account true subset condition

hyper = (nfilter, maxpool_p, learning_r)

print(starting_seed)
print(n_sub)
print(n_cells)
print(blast_perc)
print(nfilter)
print(maxpool_p)
print(learning_r)
print(labels)


# code

In [ ]:
%%time

data_folder = f'{fixed_path}B-ALL_Datasets'
extension = '*.csv'

multiple_donations, ALL_DATASETS = load_data(data_path = data_folder, ext = extension)#, remove_control = True)



In [ ]:

tot_perc_list = show_blast_distribution_perc(ALL_DATASETS, multiple_donations, return_perc = True)
print(tot_perc_list)

In [ ]:

full_LOPOCV_dicts = generate_LOPOCV_dicts(multiple_donations, ALL_DATASETS)
LOPOCV_patients_folds = generate_LOPOCV_folds(full_LOPOCV_dicts, ALL_DATASETS, starting_seed)

# Final Run

In [ ]:
# %%time

start_lopocv_fold = 0
end_lopocv_fold = 0

exp = 'Trial_5_fix_code_AS_single_split'

save_memory = False # set True to save RAM memory
grid = True

weights_outdir = f'{config_save_dir}/weights'
os.makedirs(weights_outdir, exist_ok=True)
full_process_time_list = []

for LOPOCV_fold_idx, (train_kfolds, test_pat) in enumerate(LOPOCV_patients_folds): # for each LOPO fold
  if LOPOCV_fold_idx >= start_lopocv_fold:

        LOPOCV_start = time.time()

        # === Import Base seed === #
        seed_load_dir = f'{cellcnn_path}/experiments/experiment_{tuning_exp}/outer_fold_{LOPOCV_fold_idx}/'

        with open(os.path.join(seed_load_dir, 'tuning_seed_info.pkl'), 'rb') as f:
                            seed_info = pickle.load(f)

        print('check seeds:', seed_info)
        training_split_seed = seed_info['final_tuning_base_seed']
        fold_base_seed = seed_info['fold_base_seed']

        base_seed = fold_base_seed
        np.random.seed(training_split_seed)

        # === Import Tuned theta* and BO thresholds === #
        tuning_load_dir = f'{cellcnn_path}/experiments/experiment_{tuning_exp}/outer_fold_{LOPOCV_fold_idx}/tuning/results'

        with open(os.path.join(tuning_load_dir, 'best_ncells.pkl'), 'rb') as f:
                            best_ncells = pickle.load(f)

        with open(os.path.join(tuning_load_dir, 'best_nsub.pkl'), 'rb') as f:
                            best_nsub = pickle.load(f)

        with open(os.path.join(tuning_load_dir, 'robust_threshold.pkl'), 'rb') as f:
                            robust_threshold = pickle.load(f)

        with open(os.path.join(tuning_load_dir, 'roc_threshold.pkl'), 'rb') as f:
                            roc_threshold = pickle.load(f)

        print(f'Theta*: {best_ncells, best_nsub}')
        print(f'ROC threshold: {roc_threshold}')
        print(f'RES threshold: {robust_threshold}')


        # === Generate Training and Validation Set === #

        train_val_patients = np.concatenate(train_kfolds[0]) # full set of patients idx
        np.random.shuffle(train_val_patients)
        n_val = max( int(round(0.2*len(train_val_patients))) ,1)
        train_donors_idx = train_val_patients[n_val:]
        val_donors_idx = train_val_patients[:n_val]

        print(f'Train patients: {train_donors_idx}')
        print(f'Validation patients: {val_donors_idx}')
        print(f'Test patient: {test_pat}')


        if save_memory:
            if LOPOCV_fold_idx != start_lopocv_fold:
                multiple_donations, ALL_DATASETS = load_data(data_path = data_folder, ext = extension)

        # === Extracts Samples for Train and Validation === #

        # first 30k seeds for tuning
        base_seed += 30000  #20k for AS generation
        base_final_training_AS_seed = base_seed

        # extraxt samples using pre-splitted indexes
        train_datasets_extracted = donation_extraction(train_donors_idx, multiple_donations, ALL_DATASETS)
        val_datasets_extracted = donation_extraction(val_donors_idx, multiple_donations, ALL_DATASETS)
        test_datasets_extracted = donation_extraction(test_pat, multiple_donations, ALL_DATASETS)


        # === Generate Artificial Samples for Training and Validation sets === #

        (new_train_datasets, new_train_y,
         new_val_datasets, new_val_y, _, _ ) = splitting_and_dataset_elaboration(train_datasets_extracted,
                                                                            val_datasets_extracted,
                                                                            test_datasets_extracted,
                                                                            n_sub, n_cells,           # number of AS, number of cells per AS
                                                                            starting_seed + LOPOCV_fold_idx, # seed for random extraction
                                                                            blast_perc = blast_perc,  # % of blast cells extracted
                                                                            per_perc = True)          # one Unhealthy AS per %


        del train_datasets_extracted
        del val_datasets_extracted
        if save_memory:
            del ALL_DATASETS
            gc.collect()

        print(f'End of Artificial Sample generation')


        trials = 2 ### 5

        # === Generate seed lists in advance === #

        train_CV_seed = base_seed + 1000
        seed_list = generate_seeds(len(LOPOCV_patients_folds)*2, seed = train_CV_seed)

        pred_CV_seed = base_seed + 2000
        tuning_prediction_seed_list = generate_seeds(trials, seed = pred_CV_seed)

        base_seed += 200
        final_orig_pred_seed = base_seed
        original_prediction_seed_list = generate_seeds(trials, seed = final_orig_pred_seed)

        base_seed += 1000
        final_rob_pred_seed = base_seed
        robust_prediction_seed_list = generate_seeds(3, seed = final_rob_pred_seed)


        try:

                # === Train Models === #

                models_lists = run_training(CellCnn, new_train_datasets, new_train_y,
                                            new_val_datasets = new_val_datasets,
                                            new_val_y = new_val_y,
                                            seed_list =  seed_list, hyper = hyper, grid = grid, labels = labels,
                                            trials = trials,
                                            cells_per_sub = best_ncells, ## Tuned ncells ##
                                            best_nsub = best_nsub,        ## Tuned nsub   ##
                                            max_epochs = 1,              ### 100,
                                            nrun = None,                  #grid search, so doesn't matter wht number
                                            generate = False,
                                            outdir = weights_outdir)


                # === Save Models === #

                for i, model in enumerate(models_lists):
                        save_models_dir = f'{cellcnn_path}/experiments/experiment_{exp}/outer_fold_{LOPOCV_fold_idx}/training/models/seed_{i}'
                        os.makedirs(save_models_dir, exist_ok=True)
                        save_models(model, save_models_dir)

                for i in range(1):
                        # === Prediction Approach: CellCNN with tau = 0.5 === #

                        if save_memory:
                            print('Load data for Test set')
                            multiple_donations, ALL_DATASETS = load_data(data_path = data_folder, ext = extension)

                        test_datasets_extracted = donation_extraction(test_pat, multiple_donations, ALL_DATASETS)

                        if save_memory:
                            del ALL_DATASETS

                        original_test_datasets, original_test_y = retrieve_labels(test_datasets_extracted, remove = True, flat = True)
                        print(f'Ground thruth labels: {original_test_y}')

                        # Test set prediction
                        original_predictions_list, original_results_list = trials_test_CellCNN_old(models_lists, original_test_datasets, original_prediction_seed_list)


                        print('Save CellCNN Prediction results')
                        save_original_dir = f'{cellcnn_path}/experiments/experiment_{exp}/outer_fold_{LOPOCV_fold_idx}/results/direct'

                        os.makedirs(save_original_dir, exist_ok=True)

                        with open(os.path.join(save_original_dir, 'original_predictions_list.pkl'), 'wb') as f:
                                        pickle.dump(original_predictions_list, f)
                        with open(os.path.join(save_original_dir, 'original_results_list.pkl'), 'wb') as f:
                                        pickle.dump(original_results_list, f)
                        with open(os.path.join(save_original_dir, 'original_test_y.pkl'), 'wb') as f:
                                        pickle.dump(original_test_y, f)


                        # === Prediction Approach: Resamplimg === #

                        print('Start Resampling prediction using ROC threshold')

                        test_resample_n = 100000 #same dimension of AS
                        k = 2 ### 100


                        per_donor_original_test_datasets, per_donor_original_test_y = retrieve_labels(test_datasets_extracted, remove = False)
                        print(f'Ground thruth labels per patient: {per_donor_original_test_y}')

                        # Test set prediction
                        (_, #test_total_labels,               # Predicted resampled subsets labels
                            test_total_pred_lists,           # Mean predictions of resampled subsets across trials (per patient)
                            test_total_trial_pred_lists,     # Lists of trial predictions of resampled subsets (per patient)
                            per_donor_resampled_test_y       # True labels of resampled subsets (per patient)
                                                    ) = test_res_pred(
                                                        models_lists,
                                                        per_donor_original_test_datasets,
                                                        test_resample_n,
                                                        k,
                                                        roc_threshold,
                                                        trials,
                                                        seed = robust_prediction_seed_list[1])


                        save_robust_dir = f'{cellcnn_path}/experiments/experiment_{exp}/outer_fold_{LOPOCV_fold_idx}/results/robust/roc'
                        os.makedirs(save_robust_dir, exist_ok=True)

                        with open(os.path.join(save_robust_dir, 'test_total_trial_pred_lists.pkl'), 'wb') as f:
                                        pickle.dump(test_total_trial_pred_lists, f)
                        with open(os.path.join(save_robust_dir, 'per_donor_resampled_test_y.pkl'), 'wb') as f:
                                        pickle.dump(per_donor_resampled_test_y, f)
                        with open(os.path.join(save_robust_dir, 'per_donor_original_test_y.pkl'), 'wb') as f:
                                        pickle.dump(per_donor_original_test_y, f)

        except Exception as e:
                        print(f"Training error: {e}")
                        traceback.print_exc()

        fold_end = time.time()
        print('')
        print(f'Time elapsed from start LOPOCV iteration: {LOPOCV_start - fold_end}')
        print('')



        #save seeds outside the ensemble
        tuning_save_dir = f'{cellcnn_path}/experiments/experiment_{exp}/outer_fold_{LOPOCV_fold_idx}/'
        os.makedirs(tuning_save_dir, exist_ok=True)


        with open(os.path.join(tuning_save_dir, 'seed_info.pkl'), 'wb') as f:
                                pickle.dump({
                                    'starting_seed': starting_seed,
                                    'LOPOCV_fold_idx': LOPOCV_fold_idx,
                                    'fold_base_seed': fold_base_seed,
                                    'base_final_training_AS_seed': base_final_training_AS_seed,
                                    'training_split_seed': training_split_seed,
                                    'final_base_seed': base_seed
                                }, f)

        elapsed_time_for_LOPOCV = time.time() - LOPOCV_start
        print(f'elapsed_time_for_LOPOCV fold {LOPOCV_fold_idx}: {elapsed_time_for_LOPOCV}')

        with open(os.path.join(tuning_save_dir, 'elapsed_time_for_LOPOCV.pkl'), 'wb') as f:
                                pickle.dump(elapsed_time_for_LOPOCV, f)

        if LOPOCV_fold_idx >= end_lopocv_fold:
            break
